# Signup Abuse — Entity Linkage

Unsupervised linkage of accounts plausibly sharing one real-world actor. Precision-first: legitimate customers must not be harmed.

## Pipeline at a glance — how the data is funneled

Every account enters; none is dropped until the very end (singletons). Each field is normalized to a key, keys that collide form edges, edges weighted by trust, and connected edges become clusters.

```
                    input/signups.csv
              15,008 accounts × 12 raw fields
                           │
        ┌──────────────────▼──────────────────┐
        │ 1  NORMALIZE  (all 15,008 kept)      │   raw field ──► identity key
        │    assert len == 15,008              │   missing/untrusted ──► None (links nobody)
        └──────────────────┬──────────────────┘
   email ─► canon inbox + disguise flag        phone ─► 10-digit NANP
   card  ─► BIN|last4     device ─► hash        ip+ua ─► joint key (neither alone)
   addr  ─► USPS street|unit|zip5               name  ─► sorted tokens
                           │
        ┌──────────────────▼──────────────────┐
        │ 2  INDEX + GROUP   value ─► [accts]  │   keep values shared by ≥2 accounts:
        │                                      │   email 204 · card 482 · device 47
        └──────────────────┬──────────────────┘   addr 62 · ip_ua 40 · name 982 · phone 2
                           │
        ┌──────────────────▼──────────────────┐
        │ 3  ADDRESS FUZZY  (block by zip)     │   unit-safe fuzzy on street name:
        │                                      │   +259 real typos kept · 84 apt-merges dropped
        └──────────────────┬──────────────────┘
                           │
        ┌──────────────────▼──────────────────┐
        │ 4  WEIGHT EDGES                      │   weight = BASE × hub_penalty, summed per pair
        │    computed BASE=log2(m/u) + strong- │   weights from Fellegi-Sunter (Appendix B);
        │    signal gate · hub-cap junk        │   meaning-based run kept as *_meaning.* validation
        └──────────────────┬──────────────────┘
                           │  keep pair if Σweight ≥ τ
                           ▼
                       978 edges
                           │
        ┌──────────────────▼──────────────────┐
        │ 5  UNION-FIND  → connected components│
        └──────────────────┬──────────────────┘
                           │
             ┌─────────────┴──────────────┐
             ▼                            ▼
      1,521 accounts               13,487 singletons
      in 692 clusters              (omitted per brief)
             │
        tier + rank
             │
   CERTAIN 204 · HIGH 36 · MEDIUM 452
             │
   clusters.csv  ·  edges.csv  ·  findings.md
```

*(Counts are from the current end-to-end run; every section below prints its own live numbers so the funnel stays honest on rerun.)*

> **What "unit-safe fuzzy" (step 3) means.** Exact matching misses typos: `803 ANDEESON DR` and `803 ANDERSON DR` are the same street with different letters. Fuzzy matching catches these by similarity score (0.93 = 93% same → match). But fuzzy is dangerous — it *also* wants to match different apartments in the same building:
>
> | pair | verdict |
> |---|---|
> | `5454 MUELLER DR` vs `5454 MUELL**L**ER DR` | ✅ typo, same place → **link** |
> | `803 ANDE**E**SON DR` vs `803 ANDERSON DR` | ✅ typo → **link** |
> | `5454 MUELLER DR **APT 18**` vs `… **APT 11**` | ❌ different unit = different household → **don't link** |
>
> **Rule:** accept a fuzzy match only when house-number **and** unit are identical and *only the street-name spelling* differs. That keeps the **259** real typos and drops the **84** apartment-neighbor merges. This is the precision guarantee in action — merging APT 18 with APT 11 would flag two innocent neighbors as one actor, exactly the harm the brief forbids.

## 1. Data coverage

Before any linkage: what do we actually have? Fill rate (is a signal present), cardinality (how discriminating), and **collision profile** (when a value IS shared, by how many accounts) — the last one decides each signal's trust tier.

In [1]:
import pandas as pd
from pathlib import Path

INPUT = Path("input")
df = pd.read_csv(INPUT / "signups.csv", dtype=str, keep_default_na=False)
N = len(df)
print(f"{N} accounts, {df.shape[1]} columns")
assert df["account_id"].is_unique, "account_id not unique"
df.head(3)

15008 accounts, 15 columns


,account_id,signup_ts,full_name,email,phone,addr_line1,addr_line2,city,state,zip,ip_address,payment_bin,card_last4,device_hash,user_agent
0,A000001,2026-03-25 00:13:26,Brandi Robinson,brandi_robinson@outlook.com,(577) 676-2256,9288 Kent Ave,,Port Daniel,LA,85318,104.29.154.211,453245,3188,d_13e0ad1624999c,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...
1,A000002,2026-03-17 03:29:09,Steven Rodriguez,stevenrodriguez75@aol.com,1-460-967-1187,5960 LONG AVE.,Apt 2A,New James,NJ,14717-2764,174.85.181.214,,,,Mozilla/5.0 (iPad; CPU OS 18_4 like Mac OS X) ...
2,A000003,2026-04-26 11:18:20,Lance Rogers,lancerogers@gmail.com,1-862-417-7104,9759 Wilson Ave,,Bowmanfort,PA,37459,205.254.166.183,542418,6898,d_3b69339ec7d222,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...


In [2]:
# fill rate + cardinality per raw column
cov = pd.DataFrame({
    "fill_rate": (df.ne("").sum() / N),
    "distinct":  df.nunique(),
}).sort_values("fill_rate", ascending=False)
cov["distinct_per_filled"] = (cov["distinct"] / df.ne("").sum()).round(3)
cov["fill_rate"] = (cov["fill_rate"] * 100).round(1).astype(str) + "%"
cov

,fill_rate,distinct,distinct_per_filled
account_id,100.0%,15008,1.000
signup_ts,100.0%,14997,0.999
full_name,100.0%,13723,0.914
email,100.0%,15008,1.000
addr_line1,100.0%,14927,0.995
city,100.0%,10476,0.698
state,100.0%,59,0.004
zip,100.0%,14024,0.934
ip_address,100.0%,14796,0.986
user_agent,100.0%,7,0.000


In [3]:
# Collision profile: when a NORMALIZED value is shared, by how many accounts?
# This — not fill rate — decides trust. A signal whose shared values are tiny
# groups (phone) is trustworthy; one whose shared values are huge (user_agent,
# non-relay IP hubs) is noise/NAT and cannot link on its own.
import re, ipaddress

relay_nets = [ipaddress.ip_network(c) for c in
              pd.read_csv(INPUT / "icloud_relay_ranges.csv")["cidr"]]
def is_relay(ip):
    try: a = ipaddress.ip_address(ip)
    except ValueError: return False
    return any(a in n for n in relay_nets)

def phone10(s):
    d = re.sub(r"\D", "", s); d = d[1:] if len(d) == 11 and d[0] == "1" else d
    return d if len(d) == 10 else None
def name_key(s):
    t = re.sub(r"[^\w\s]", " ", s.lower()).split()
    return " ".join(sorted(t)) or None

sig = pd.DataFrame({
    "phone":       df["phone"].map(phone10),
    "card_bin+l4": (df["payment_bin"] + "|" + df["card_last4"]).where(df["payment_bin"].ne("") & df["card_last4"].ne("")),
    "bin_alone":   df["payment_bin"].replace("", None),
    "device":      df["device_hash"].replace("", None),
    "ip_nonrelay": df["ip_address"].where(~df["ip_address"].map(is_relay)).replace("", None),
    "ip_relay":    df["ip_address"].where(df["ip_address"].map(is_relay)).replace("", None),
    "user_agent":  df["user_agent"].replace("", None),
    "name":        df["full_name"].map(name_key),
    "zip5":        df["zip"].str.replace(r"\D", "", regex=True).str[:5].replace("", None),
})

def collision(col):
    vc = col.value_counts()
    shared = vc[vc >= 2]
    return pd.Series({
        "shared_vals":   len(shared),
        "accts_shared":  int(shared.sum()),
        "pct_accts":     f"{shared.sum()/N:.1%}",
        "max_hub":       int(vc.max()) if len(vc) else 0,
        "top5_hubs":     list(shared.head(5).values),
    })

prof = pd.DataFrame({c: collision(sig[c]) for c in sig}).T
prof

,shared_vals,accts_shared,pct_accts,max_hub,top5_hubs
phone,2,7,0.0%,4,"[4, 3]"
card_bin+l4,482,1011,6.7%,18,"[18, 10, 8, 3, 3]"
bin_alone,12,10920,72.8%,965,"[965, 946, 943, 924, 922]"
device,47,111,0.7%,3,"[3, 3, 3, 3, 3]"
ip_nonrelay,6,217,1.4%,46,"[46, 43, 41, 33, 32]"
ip_relay,1,2,0.0%,2,[2]
user_agent,7,15008,100.0%,2204,"[2204, 2185, 2172, 2162, 2110]"
name,982,2278,15.2%,12,"[12, 7, 7, 7, 7]"
zip5,1149,2586,17.2%,49,"[49, 26, 21, 20, 17]"


### Phone row:

- shared_vals = 2 → only 2 phone numbers show up on more than one account. Every other phone = unique to one account.
- accts_shared = 7 → those 2 numbers cover 7 accounts total.
- pct_accts = 0.0% → 7 out of 15008 = basically nothing.
- max_hub = 4 → the bigger of the 2 numbers is shared by 4 accounts.
- top5_hubs = [4, 3] → the group sizes: one number on 4 accounts, other on 3. (4+3=7, matches.)

**Meaning:** *Almost nobody reuses a phone. When they do, tiny group (3-4). So a shared phone = strong hint of same person. Rare, but trustworthy.*

In [4]:
# Email is 100% unique RAW — the links only appear after canonicalizing the inbox.
# Two grades: same canonical inbox = same address reused; same inbox reached via
# DIFFERENT raw spellings on a provider that ignores dots/+tags = deliberate
# disguise (one person, provably) — the strongest single signal we have.
GMAIL = {"gmail.com", "googlemail.com"}
ALIASING = GMAIL | {"outlook.com", "hotmail.com", "live.com", "yahoo.com",
                    "proton.me", "icloud.com", "fastmail.com"}

def email_keys(s):
    e = s.strip().lower()
    if e.count("@") != 1: return None, None, False
    local, dom = e.split("@")
    if not local or not dom: return None, None, False
    if dom == "googlemail.com": dom = "gmail.com"
    aliasable = dom in ALIASING
    clocal = local.split("+")[0]
    if dom in GMAIL: clocal = clocal.replace(".", "")
    if not clocal: return None, None, False
    return f"{clocal}@{dom}", f"{local}@{dom}", aliasable   # canonical, raw, aliasable

ek = df["email"].map(email_keys)
em = pd.DataFrame(ek.tolist(), columns=["canon", "raw", "aliasable"])

g = em.dropna(subset=["canon"]).groupby("canon")
shared = g.size()[g.size() >= 2]
# of the shared inboxes, how many are disguise (>1 raw spelling, aliasable provider)
disguise = g.agg(n=("raw", "size"), n_raw=("raw", "nunique"), aliasable=("aliasable", "all"))
disguise = disguise[(disguise.n >= 2) & (disguise.n_raw > 1) & disguise.aliasable]
print(f"canonical inboxes shared by >=2 accts : {len(shared)}  ({int(shared.sum())} accounts)")
print(f"  of which DISGUISE (diff spellings)  : {len(disguise)}  ({int(disguise.n.sum())} accounts)")
disguise.sort_values("n", ascending=False).head(10)

canonical inboxes shared by >=2 accts : 204  (473 accounts)
  of which DISGUISE (diff spellings)  : 204  (473 accounts)


,n,n_raw,aliasable
canon,,,
jamiejackson@gmail.com,9,9,True
teresahodges@gmail.com,8,8,True
tsmith@gmail.com,6,6,True
mjohnson@gmail.com,5,5,True
dgarcia@gmail.com,4,4,True
dmd@gmail.com,4,4,True
jbrown@gmail.com,4,4,True
msmith@gmail.com,4,4,True
mmd@gmail.com,4,4,True


**Example — one inbox, 9 disguised spellings.** `jamiejackson@gmail.com` is the busiest. Gmail ignores dots and `+tags`, so every row below lands in the *same* mailbox — one person, 9 accounts, cosmetically varied to look distinct.

In [5]:
# Show the raw as-entered emails that all canonicalize to jamiejackson@gmail.com
mask = em["canon"] == "jamiejackson@gmail.com"
example = df.loc[mask.values, ["account_id", "email", "full_name", "signup_ts"]].copy()
example["canonical_inbox"] = "jamiejackson@gmail.com"
print(f"{len(example)} accounts, all delivering to the same Gmail inbox:\n")
example.reset_index(drop=True)

9 accounts, all delivering to the same Gmail inbox:



,account_id,email,full_name,signup_ts,canonical_inbox
0,A001308,jamiejacks.on@gmail.com,marcus webb,2026-03-04 10:48:42,jamiejackson@gmail.com
1,A002691,jamiejackson+new@gmail.com,marcus webb,2026-03-06 20:11:26,jamiejackson@gmail.com
2,A002938,jamiejacks.on+promo@gmail.com,marcus webb,2026-03-14 10:17:39,jamiejackson@gmail.com
3,A007585,jamiejackson@gmail.com,"Webb, Marcus",2026-03-12 12:57:52,jamiejackson@gmail.com
4,A007784,ja.miejackson+shop@gmail.com,Marcus X. Webb,2026-03-14 01:51:43,jamiejackson@gmail.com
5,A008340,jamiejackson+52@gmail.com,"Webb, Marcus",2026-03-04 05:20:41,jamiejackson@gmail.com
6,A009291,jamiejackso.n+shop@gmail.com,Marcus L. Webb,2026-03-17 06:47:40,jamiejackson@gmail.com
7,A011275,jamiejackson+offers@gmail.com,Marcus M. Webb,2026-03-04 12:25:13,jamiejackson@gmail.com
8,A012022,jamie.jackson@gmail.com,M. Webb,2026-03-12 04:11:06,jamiejackson@gmail.com


## 2. Normalization — one canonical key per signal

Collapse each raw column to the value that identifies a *person*, so exact-matching keys does the linking. Reuses the helpers above (`email_keys`, `phone10`, `name_key`, `is_relay`) plus an address normalizer. Each account → one row of canonical keys; untrustworthy/missing values become `None` and link nobody.

In [6]:
# --- address + phone-quality normalizers ---
USPS = {"AVENUE":"AVE","AV":"AVE","BOULEVARD":"BLVD","STREET":"ST","STR":"ST",
        "ROAD":"RD","LANE":"LN","DRIVE":"DR","COURT":"CT","CIRCLE":"CIR",
        "PLACE":"PL","HIGHWAY":"HWY","PARKWAY":"PKWY"}
DIRS = {"NORTH":"N","SOUTH":"S","EAST":"E","WEST":"W","NORTHEAST":"NE",
        "NORTHWEST":"NW","SOUTHEAST":"SE","SOUTHWEST":"SW"}
FAKE_PHONES = {"0000000000","1234567890","5555555555"}

def zip5(s):
    z = re.sub(r"\D", "", s)[:5]
    return z if len(z) == 5 else None

def addr_key(line1, line2, z):
    z = zip5(z)
    toks = re.sub(r"[^\w\s]", " ", line1.upper()).split()
    toks = [USPS.get(DIRS.get(t, t), DIRS.get(t, t)) for t in toks]
    street = " ".join(toks)
    unit = " ".join(re.sub(r"[^\w\s]", " ", line2.upper()).split())
    return f"{street}|{unit}|{z}" if street and z else None

def phone_ok(s):                      # phone10 + drop invalid NANP / fakes
    d = phone10(s)
    if d is None or d[0] in "01" or d in FAKE_PHONES or len(set(d)) == 1:
        return None
    return d

# --- assemble one canonical-key row per account ---
# email disguise is detected later from email_raw spellings; here we just carry
# the canonical inbox + the raw + whether the provider ignores dots/+tags.
K = pd.DataFrame({"account_id": df["account_id"]})
K["email"]           = em["canon"]
K["email_raw"]       = em["raw"]
K["email_aliasable"] = em["aliasable"]
K["phone"]  = df["phone"].map(phone_ok)
K["card"]   = (df["payment_bin"] + "|" + df["card_last4"]).where(df["payment_bin"].ne("") & df["card_last4"].ne(""))
K["device"] = df["device_hash"].replace("", None)
# ip and user_agent are noise ALONE (ip hub 46, ua hub 2204). Joined they mean
# "same network AND same browser" -> tight (joint median hub 5). So the signal is
# the JOINT key, never either field alone. Relay IPs dropped first.
_relay = df["ip_address"].map(is_relay)
_ip    = df["ip_address"].where(~_relay).replace("", None)
K["ip_ua"]  = (_ip + " || " + df["user_agent"]).where(_ip.notna())
K["addr"]   = [addr_key(a, b, z) for a, b, z in zip(df["addr_line1"], df["addr_line2"], df["zip"])]
K["zip5"]   = df["zip"].map(zip5)
K["name"]   = df["full_name"].map(name_key)

# full-data invariant: one row per account, no signup dropped in normalization.
# Missing/untrustworthy fields become None (link nobody) but the ACCOUNT stays.
assert len(K) == N and K["account_id"].is_unique, "normalization lost accounts!"

print(f"normalized all {len(K)} accounts (no rows dropped). non-null keys per signal:")
print(K.drop(columns=["account_id","email_raw","email_aliasable"]).notna().sum())
K.head(3)

normalized all 15008 accounts (no rows dropped). non-null keys per signal:
email     15008
phone     11740
card      10920
device     6017
ip_ua     14302
addr      15008
zip5      15008
name      15008
dtype: int64


,account_id,email,email_raw,email_aliasable,phone,card,device,ip_ua,addr,zip5,name
0,A000001,brandi_robinson@outlook.com,brandi_robinson@outlook.com,True,5776762256,453245|3188,d_13e0ad1624999c,NaN,9288 KENT AVE||85318,85318,brandi robinson
1,A000002,stevenrodriguez75@aol.com,stevenrodriguez75@aol.com,False,4609671187,NaN,NaN,174.85.181.214 || Mozilla/5.0 (iPad; CPU OS 18...,5960 LONG AVE|APT 2A|14717,14717,rodriguez steven
2,A000003,lancerogers@gmail.com,lancerogers@gmail.com,True,8624177104,542418|6898,d_3b69339ec7d222,205.254.166.183 || Mozilla/5.0 (Windows NT 10....,9759 WILSON AVE||37459,37459,lance rogers


## 3. Address fuzzy layer — does it earn its keep?

Hard-normalized `addr` exact-match already links 229 accounts. Fuzzy matching would catch *transposition* typos that survive normalization (`WILSON` vs `WISLON`). But fuzzy across 15k rows is 112M pairs — so we **block by zip5** (blocks ≤49) and only fuzzy-compare within a block, using stdlib `difflib` (no new dependency; blocks are tiny). Before wiring it into the model we measure: **how many extra account-pairs does fuzzy find beyond exact?** If negligible → cut it (YAGNI).

In [7]:
from difflib import SequenceMatcher
from itertools import combinations

FUZZY_THRESH = 0.90
# street part only (drop the |unit|zip tail) so we compare the actual line1
street = K["addr"].dropna().str.split("|").str[0]
by_zip = K.loc[street.index].assign(street=street).groupby("zip5")

exact_pairs, fuzzy_only_pairs = set(), []
for z, grp in by_zip:
    if len(grp) < 2:
        continue
    for (i, a), (j, b) in combinations(grp.iterrows(), 2):
        if a["street"] == b["street"]:
            exact_pairs.add((a["account_id"], b["account_id"]))
        elif SequenceMatcher(None, a["street"], b["street"]).ratio() >= FUZZY_THRESH:
            fuzzy_only_pairs.append((a["account_id"], b["account_id"],
                                     a["street"], b["street"]))

print(f"exact addr pairs (within-zip): {len(exact_pairs)}")
print(f"fuzzy-ONLY extra pairs        : {len(fuzzy_only_pairs)}")
pd.DataFrame(fuzzy_only_pairs, columns=["acct_a","acct_b","street_a","street_b"]).head(12)

exact addr pairs (within-zip): 933
fuzzy-ONLY extra pairs        : 343


,acct_a,acct_b,street_a,street_b
0,A002744,A008017,5454 MUELLER DR APT 18,5454 MUELLER DR APT 11
1,A002744,A012657,5454 MUELLER DR APT 18,5454 MUELLER DR APT 25
2,A002746,A005664,5454 MUELLER DR,5454 MUELLER DR 10
3,A002746,A013372,5454 MUELLER DR,5454 MUELLLER DR
4,A002746,A014192,5454 MUELLER DR,5454 MUELLER DR 60
5,A003002,A005664,5454 MUELLER DR,5454 MUELLER DR 10
6,A003002,A013372,5454 MUELLER DR,5454 MUELLLER DR
7,A003002,A014192,5454 MUELLER DR,5454 MUELLER DR 60
8,A003894,A005664,5454 MUELLER DR,5454 MUELLER DR 10
9,A003894,A013372,5454 MUELLER DR,5454 MUELLLER DR


In [8]:
# Classify the fuzzy-only pairs: is the difference a street-NAME typo (safe to
# link) or a different UNIT/number in the same building (must NOT link)?
def unit_tokens(s):
    # trailing apartment/unit token(s): digits or APT/UNIT/# markers
    return tuple(t for t in s.split() if t.isdigit() or t in {"APT","UNIT","STE","#"})

safe_typo, unit_mismatch = 0, 0
for a, b, sa, sb in fuzzy_only_pairs:
    if unit_tokens(sa) == unit_tokens(sb):
        safe_typo += 1          # same unit, only spelling differs -> real typo
    else:
        unit_mismatch += 1      # different unit/number -> different household
print(f"fuzzy-only pairs total : {len(fuzzy_only_pairs)}")
print(f"  same-unit typo (safe): {safe_typo}")
print(f"  UNIT MISMATCH (harmful, different household): {unit_mismatch}")

fuzzy-only pairs total : 343
  same-unit typo (safe): 259
  UNIT MISMATCH (harmful, different household): 84


**Verdict.** Fuzzy finds 343 extra pairs: **259 safe** (same house number + same unit, only the street *name* misspelled — `MUELLER`/`MUELLLER`) and **84 harmful** (different apartment/unit in one building = different households). So we keep fuzzy but make it **unit-safe**: link only when house-number and unit tokens match exactly and the street *name* is a near-match. This captures the real typos your idea targets and drops the apartment-merge false positives. Address stays **corroborate-only** regardless — it raises weight on an already-linked pair but never links a pair alone.

In [9]:
# Build an address GROUP id per account = union of exact-key matches and
# unit-safe fuzzy matches. Downstream this behaves like any other shared key
# (group id is the "value"), so hub-cap / IDF apply uniformly.
class UF:
    def __init__(self): self.p = {}
    def find(self, x):
        self.p.setdefault(x, x)
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[max(ra, rb)] = min(ra, rb)

uf_addr = UF()
# exact: same full addr key
for key, grp in K.dropna(subset=["addr"]).groupby("addr"):
    ids = grp["account_id"].tolist()
    for other in ids[1:]:
        uf_addr.union(ids[0], other)
# unit-safe fuzzy: same house# + unit tokens, near-match street name
for a, b, sa, sb in fuzzy_only_pairs:
    if unit_tokens(sa) == unit_tokens(sb):
        uf_addr.union(a, b)

# group id only where an account actually joined a >=2 group
root = {aid: uf_addr.find(aid) for aid in K.loc[K["addr"].notna(), "account_id"]}
gsize = pd.Series(list(root.values())).value_counts()
K["addr_group"] = K["account_id"].map(
    lambda a: root.get(a) if a in root and gsize[root[a]] >= 2 else None)

n_grp = K["addr_group"].nunique()
print(f"address groups (>=2 accts): {n_grp}, covering {K['addr_group'].notna().sum()} accounts")
print(f"biggest group size: {int(gsize.max())}")

address groups (>=2 accts): 62, covering 242 accounts
biggest group size: 33


## 4. Weighted edges → clusters

Each shared key drops weight onto every pair that shares it: `weight = BASE[signal] × hub_penalty(group_size)`. A pair's weights **add up**; keep the pair as an edge if the total ≥ τ (=4); connected edges = a cluster.

**BASE does two separate jobs — don't conflate them:**

| job | | |
|---|---|---|
| **LINK weight** (gates edges) | `email_alias 8` · `email/card/device/phone 4` · `addr 2.5` · `name 1.5` · `ip+ua 1.5` | all 4 strong = 4 so **each links alone**; corroborate < τ so they **never link alone** |
| **RANK order** (headline signal only) | email > card > device > phone > addr > name > ip+ua | breaks ties for reporting; not a weight |

- **BASE is a trust *prior* from field meaning**, not a computed number. We *tested* deriving it from data (Fellegi–Sunter `log₂(m/u)`, labels bootstrapped from the disguise rings) — it ranks *name above card* and is blind to false positives, so we keep the meaning prior. Full experiment and reasoning in **Appendix A** at the bottom.
- **hub_penalty** decays weight as a value is shared by more accounts (many sharers = infrastructure). **HUB_CAP** hard-drops oversized groups (card>8, addr>12, ip+ua>10) — a card on 18 or a building on 31 is collision, not an actor. This is the data-driven, *per-value* part that a single per-signal BASE can't capture.
- **ip and user_agent are gone as standalone signals** — replaced by the single joint `ip||ua` key (either alone is noise; together = same network + same browser).

In [10]:
from collections import defaultdict

# --- trust weights: BASE does two jobs, kept separate ---
# LINK WEIGHT gates edges. All 4 strong share weight 4 so each links ALONE;
# corroborate (addr/name/ip_ua) sit below TAU so they never link a pair alone.
BASE = {"email_alias": 8.0,                                        # certainty
        "email": 4.0, "card": 4.0, "device": 4.0, "phone": 4.0,   # strong (link alone)
        "addr": 2.5, "name": 1.5, "ip_ua": 1.5}                   # corroborate-only (< TAU)
# RANK only breaks ties for the headline signal — order among equals, not a weight.
# device >= phone: device is cleanest (hub <=3); phone is thin (7 accts)/recyclable.
RANK = {"email_alias": 7, "email": 6, "card": 5, "device": 4,
        "phone": 3, "addr": 2, "name": 1, "ip_ua": 0}
SOFT = {"email_alias": 8, "email": 4, "card": 3, "device": 3,
        "phone": 3, "addr": 4, "name": 3, "ip_ua": 4}
HUB_CAP = {"card": 8, "addr": 12, "ip_ua": 10, "name": 10}   # suppress groups larger than this
TAU = 4.0                                                    # min combined weight to keep an edge

# A shared canonical inbox is ONE person no matter how many accounts use it, so
# email is never a "hub" -> no penalty. Penalty only applies to values many
# DIFFERENT people can share (NAT IP, common name, apartment building).
NO_HUB_PENALTY = {"email_alias", "email"}
def hub_penalty(df, sig):
    if sig in NO_HUB_PENALTY:
        return 1.0
    return 1.0 / (1.0 + max(0, df - SOFT[sig]))

# --- inverted index: signal value -> [account_ids] ---
STRAIGHT = {"phone": "phone", "card": "card", "device": "device",
            "ip_ua": "ip_ua", "addr": "addr_group", "name": "name"}
index = {sig: defaultdict(list) for sig in STRAIGHT}
for sig, col in STRAIGHT.items():
    for aid, val in zip(K["account_id"], K[col]):
        if val is not None and not (isinstance(val, float) and pd.isna(val)):
            index[sig][val].append(aid)

pair_w = defaultdict(float)
pair_sig = defaultdict(list)
suppressed = []

def add_group(sig, val, members):
    df = len(members)
    if df < 2:
        return
    if sig in HUB_CAP and df > HUB_CAP[sig]:
        suppressed.append((sig, val, df))
        return
    w = BASE[sig] * hub_penalty(df, sig)
    members = sorted(members)
    for i in range(len(members)):
        for j in range(i + 1, len(members)):
            p = (members[i], members[j])
            pair_w[p] += w
            pair_sig[p].append((sig, w))

# email: split disguise (email_alias) vs plain reuse (email)
email_members = defaultdict(list)
for aid, canon in zip(K["account_id"], K["email"]):
    if pd.notna(canon):
        email_members[canon].append(aid)
for canon, members in email_members.items():
    if len(members) < 2:
        continue
    sub = K.set_index("account_id").loc[members]
    aliasable = sub["email_aliasable"].all()
    n_raw = sub["email_raw"].nunique()
    sig = "email_alias" if (aliasable and n_raw > 1) else "email"
    add_group(sig, canon, members)

# straight keyed signals
for sig, idx in index.items():
    for val, members in idx.items():
        add_group(sig, val, members)

print(f"candidate pairs: {len(pair_w)}   suppressed hub groups: {len(suppressed)}")
print(f"pairs >= TAU ({TAU}): {sum(w >= TAU for w in pair_w.values())}")

candidate pairs: 3544   suppressed hub groups: 6
pairs >= TAU (4.0): 978


In [11]:
# threshold edges -> union-find -> connected components
uf = UF()
kept = []
for (a, b), w in sorted(pair_w.items()):
    if w >= TAU:
        uf.union(a, b)
        kept.append((a, b, round(w, 2), pair_sig[(a, b)]))

comp = defaultdict(list)
for aid in K["account_id"]:
    r = uf.find(aid)
    if r != aid or aid in uf.p:      # only accounts that joined something
        comp[r].append(aid)
clusters = {r: sorted(m) for r, m in comp.items() if len(m) > 1}

# signals present per cluster (from its kept edges)
edge_sigs = defaultdict(set)
for a, b, w, sigs in kept:
    r = uf.find(a)
    for s, _ in sigs:
        edge_sigs[r].add(s)

print(f"kept edges: {len(kept)}   clusters (>=2 accts): {len(clusters)}")
print(f"accounts linked: {sum(len(m) for m in clusters.values())}")
sizes = sorted((len(m) for m in clusters.values()), reverse=True)
print(f"cluster size distribution top10: {sizes[:10]}")

kept edges: 978   clusters (>=2 accts): 692
accounts linked: 1519
cluster size distribution top10: [9, 8, 7, 6, 5, 5, 5, 5, 4, 4]


## 5. Tier, rank, and outputs

Each cluster gets a confidence tier from the signals that link it:

- **CERTAIN** — carries an email-disguise link (provably one inbox, deliberately varied spellings).
- **HIGH** — ≥2 independent signal types including a strong one.
- **MEDIUM** — a single strong signal (one shared device/card/phone/inbox).
- **LOW** — only corroborate-only signals stacked (weaker; review before action).
- **REVIEW** — oversized component with no disguise proof (possible over-merge).

Then write the deliverables: `output/clusters.csv` (account→cluster), `output/edges.csv` (audit trail), and ranked findings.

In [12]:
STRONG = {"email", "card", "device", "phone"}
REVIEW_SIZE = 25
Kidx = K.set_index("account_id")

def shared_value(members, col):
    vc = Kidx.loc[members, col].dropna().value_counts()
    return vc.index[0] if len(vc) else None

rows = []
for r, members in clusters.items():
    sigs = edge_sigs[r]
    cedges = [e for e in kept if uf.find(e[0]) == r]
    max_w = max(w for _, _, w, _ in cedges)
    n = len(members)
    has_alias = "email_alias" in sigs
    has_strong = bool(sigs & STRONG) or has_alias
    n_types = len(sigs)
    if has_alias:                    tier = "CERTAIN"
    elif has_strong and n_types >= 2: tier = "HIGH"
    elif has_strong:                 tier = "MEDIUM"
    else:                            tier = "LOW"
    if n > REVIEW_SIZE and not has_alias: tier = "REVIEW"
    # linking values for the narrative
    colmap = {"email_alias": "email", "email": "email", "phone": "phone",
              "card": "card", "device": "device", "ip_ua": "ip_ua",
              "addr": "addr", "name": "name"}
    links = {s: shared_value(members, colmap[s]) for s in sorted(sigs)}
    dominant = max(sigs, key=lambda s: RANK[s])     # RANK, not BASE (strong all tie at 4)
    confidence = round(max_w * (1 + 0.4 * (n_types - 1)), 2)
    rows.append({"members": members, "size": n, "tier": tier,
                 "confidence": confidence, "dominant_signal": dominant,
                 "signal_types": ",".join(sorted(sigs)),
                 "linking_values": "; ".join(f"{s}={v}" for s, v in links.items() if v)})

order = {"REVIEW": 5, "CERTAIN": 4, "HIGH": 3, "MEDIUM": 2, "LOW": 1}
rows.sort(key=lambda c: (order[c["tier"]], c["confidence"], c["size"]), reverse=True)
for i, c in enumerate(rows, 1):
    c["cluster_id"] = f"C{i:05d}"

from collections import Counter
print("tier breakdown:", dict(Counter(c["tier"] for c in rows)))
pd.DataFrame([{k: c[k] for k in ("cluster_id","size","tier","confidence","dominant_signal","linking_values")}
              for c in rows[:10]])

tier breakdown: {'CERTAIN': 204, 'HIGH': 35, 'MEDIUM': 453}


,cluster_id,size,tier,confidence,dominant_signal,linking_values
0,C00001,7,CERTAIN,17.1,email_alias,card=426684|6687; email_alias=tsmith@gmail.com...
1,C00002,4,CERTAIN,17.1,email_alias,card=414720|8012; email_alias=richardhenderson...
2,C00003,3,CERTAIN,17.1,email_alias,card=414720|0712; email_alias=shawn_jones@gmai...
3,C00004,3,CERTAIN,17.1,email_alias,card=551149|6123; email_alias=maryjames@gmail....
4,C00005,3,CERTAIN,17.1,email_alias,card=542418|8329; email_alias=tina_wallace@gma...
5,C00006,3,CERTAIN,17.1,email_alias,card=374245|6640; email_alias=ryansmith@gmail....
6,C00007,9,CERTAIN,16.8,email_alias,device=d_220bd46d2ea158; email_alias=jamiejack...
7,C00008,8,CERTAIN,16.8,email_alias,email_alias=teresahodges@gmail.com; phone=7704...
8,C00009,3,CERTAIN,15.3,email_alias,card=411111|8018; email_alias=patriciajohnson@...
9,C00010,5,CERTAIN,14.4,email_alias,addr=4910 W MORRIS AVE 2C||36166; device=d_580...


In [13]:
import csv
OUT = Path("output"); OUT.mkdir(exist_ok=True)

# clusters.csv  (account_id -> cluster_id)
with open(OUT / "clusters_meaning.csv", "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(["account_id", "cluster_id", "cluster_size", "confidence_tier",
                 "dominant_signal", "linking_values"])
    for c in rows:
        for aid in c["members"]:
            wr.writerow([aid, c["cluster_id"], c["size"], c["tier"],
                         c["dominant_signal"], c["linking_values"]])

# edges.csv  (audit trail)
with open(OUT / "edges_meaning.csv", "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(["account_a", "account_b", "combined_weight", "signals"])
    for a, b, w, sigs in sorted(kept):
        wr.writerow([a, b, w, "|".join(f"{s}:{round(sw,2)}" for s, sw in sigs)])

# ranked findings (meaning-based, VALIDATION)
with open(OUT / "findings_meaning.md", "w") as f:
    f.write(f"# Ranked findings\n\n{N} accounts | {len(rows)} clusters | "
            f"{sum(c['size'] for c in rows)} accounts linked\n\n"
            f"Tiers: {dict(Counter(c['tier'] for c in rows))}\n\n")
    for c in rows[:15]:
        f.write(f"### {c['cluster_id']} — {c['size']} accounts — {c['tier']} "
                f"(confidence {c['confidence']})\n"
                f"- Linked by: {c['linking_values']}\n"
                f"- Members: {', '.join(c['members'][:12])}"
                f"{' …' if c['size'] > 12 else ''}\n\n")

print("wrote:", *[p.name for p in OUT.iterdir()])
print(open(OUT / "findings_meaning.md").read()[:900])

wrote: clusters.csv findings_meaning.md clusters_meaning.csv findings.md edges_meaning.csv edges.csv
# Ranked findings

15008 accounts | 692 clusters | 1519 accounts linked

Tiers: {'CERTAIN': 204, 'HIGH': 35, 'MEDIUM': 453}

### C00001 — 7 accounts — CERTAIN (confidence 17.1)
- Linked by: card=426684|6687; email_alias=tsmith@gmail.com; name=smith tyler
- Members: A000750, A001394, A002118, A003871, A004992, A008904, A012395

### C00002 — 4 accounts — CERTAIN (confidence 17.1)
- Linked by: card=414720|8012; email_alias=richardhenderson@gmail.com; name=henderson richard
- Members: A000078, A008038, A008771, A009565

### C00003 — 3 accounts — CERTAIN (confidence 17.1)
- Linked by: card=414720|0712; email_alias=shawn_jones@gmail.com; name=jones shawn
- Members: A000755, A009958, A013226

### C00004 — 3 accounts — CERTAIN (confidence 17.1)
- Linked by: card=551149|6123; email_alias=maryjames@gmail.com; name=james mary
- Members: A001563, A009189, A010950

### C00005 — 3 accounts — CERTAIN 

## 6. Worked example — one cluster, every transform

Trace the Cabrera cluster (member `A004729`) end to end: raw fields → canonical keys → which values collide → per-group weight → per-pair sum → why the cluster forms. This is the whole pipeline on one case.

In [14]:
# locate the cluster containing A004729, then narrate it
anchor = "A004729"
root = uf.find(anchor)
mem = clusters[root]
print(f"cluster of {anchor}: {mem}\n")

print("STEP 1 — canonical keys per account:")
print(Kidx.loc[mem, ["email", "phone", "card", "device", "addr", "ip_ua"]].to_string())

print("\nSTEP 2/3 — kept edges (pair -> signals with weights):")
for a, b, w, sigs in kept:
    if uf.find(a) == root:
        detail = " + ".join(f"{s}({sw:.1f})" for s, sw in sigs)
        print(f"  {a}-{b}: {detail} = {w}  {'>= TAU' if w >= TAU else ''}")

c = next(c for c in rows if anchor in c["members"])
print(f"\nSTEP 4 — result: {c['cluster_id']} | size {c['size']} | {c['tier']} "
      f"| headline={c['dominant_signal']} | confidence {c['confidence']}")
print(f"         linked by: {c['linking_values']}")

cluster of A004729: ['A000047', 'A003535', 'A004729', 'A006216']

STEP 1 — canonical keys per account:
                               email       phone         card            device                     addr                                                                                                                                            ip_ua
account_id                                                                                                                                                                                                                                               
A000047     tammycabrera78@gmail.com  4899927688          NaN  d_9cfdc4a98f2cae   803 ANDEESON DR||15627             24.64.212.1 || Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36
A003535        michaelmd60@yahoo.com  7664313811  440066|3240  d_3dd4716499f460  6795 SE CURRY LN||27971  73.234.43.208 || Mozilla/5.0 (iPad; CPU O

In [15]:
# Live funnel recap — numbers pulled from the run, so the diagram up top stays honest.
grp = {s: sum(1 for m in index[s].values() if len(m) >= 2) for s in index}
grp["email"] = sum(1 for m in email_members.values() if len(m) >= 2)
linked = sum(len(m) for m in clusters.values())
tiers = Counter(c["tier"] for c in rows)
print("DATA FUNNEL (live)")
print(f"  raw accounts .................. {N:>6}")
print(f"  normalized (kept) ............. {len(K):>6}   (assert len==N passed)")
print(f"  signal groups >=2 accts ....... email {grp['email']} · card {grp['card']} · "
      f"device {grp['device']} · addr {grp['addr']} · ip_ua {grp['ip_ua']} · name {grp['name']} · phone {grp['phone']}")
print(f"  candidate pairs ............... {len(pair_w):>6}")
print(f"  hub groups suppressed ......... {len(suppressed):>6}")
print(f"  edges kept (>= TAU {TAU}) ........ {len(kept):>6}")
print(f"  clusters ...................... {len(clusters):>6}")
print(f"  accounts linked ............... {linked:>6}   singletons dropped {N - linked}")
print(f"  tiers ......................... {dict(tiers)}")

DATA FUNNEL (live)
  raw accounts ..................  15008
  normalized (kept) .............  15008   (assert len==N passed)
  signal groups >=2 accts ....... email 204 · card 482 · device 47 · addr 62 · ip_ua 40 · name 982 · phone 2
  candidate pairs ...............   3544
  hub groups suppressed .........      6
  edges kept (>= TAU 4.0) ........    978
  clusters ......................    692
  accounts linked ...............   1519   singletons dropped 13489
  tiers ......................... {'CERTAIN': 204, 'HIGH': 35, 'MEDIUM': 453}


## Appendix A — Can BASE be computed from data? (why we keep a meaning-based prior)

The obvious objection to hand-set BASE weights: *derive them from the data instead.* We can — and did. The result is instructive, and the reason we don't deploy it is the core of the whole precision-first argument. This is the classic **Fellegi–Sunter** record-linkage weight, `BASE = log₂(m / u)`:

- **`m` = P(field matches | the two accounts ARE the same actor)** — "does the same person reuse this?" Needs ground-truth same-actor labels. We bootstrap them from the **disguise inboxes** (same canonical inbox, different spellings = provably one person).
- **`u` = P(field matches | two RANDOM accounts)** — "do strangers collide on it by chance?" Just how concentrated the values are: `Σ (count/n)²`.

**The four steps (worked on `card`):**
1. **Labels:** every pair of accounts inside a disguise ring → 387 same-actor pairs.
2. **`m`:** of the 387, 215 had a card on both; 36 matched → `m = 36/215 = 0.167`.
3. **`u`:** 10,920 accounts have a card → `u = Σ(count/10920)² = 1.04e-04`.
4. **BASE:** `log₂(0.167 / 0.000104) = log₂(1610) = 10.65`.

**Why we do NOT ship these weights** (see the table the next cell prints):
- It ranks **name (11.44) above card (10.65)** — because within these rings one person reuses their name, so `m` is high. But this only measures **within-ring reuse (recall)**. It is **blind to false positives**: `u` is an *average*, and the average name is rare, so it can't see that the *specific* value `"smith john"` is shared by hundreds of unrelated strangers. Deploying name at weight 11 would merge every John Smith in the file.
- The label set is **all true-positives, no negatives** — so it structurally cannot learn the false-positive cost, which is the graded failure mode.
- It correctly zeroes `ip_ua` (never matches within a ring → useless as a same-actor key), confirming our weak tier.

**Conclusion:** the computed weight optimizes recall on known rings; precision-first needs the opposite. So BASE stays a **meaning prior** (what could this value falsely collide on?) and the data-driven part lives **per-value** in `hub_penalty`/`HUB_CAP`, which *can* see that `"smith john"` is a hub. The computation below is a **validator**, not the deployed setter.

In [16]:
import math

# STEP 1 — labels: pairs inside a disguise inbox = provably same actor
Ki = K.reset_index(drop=True)
canon_of = K["email"]
same_actor_pairs = []
for canon, members in email_members.items():
    if len(members) < 2:
        continue
    sub = K.set_index("account_id").loc[members]
    if sub["email_aliasable"].all() and sub["email_raw"].nunique() > 1:   # disguise
        idx = [K.index[K["account_id"] == a][0] for a in members]
        same_actor_pairs += list(combinations(idx, 2))

def m_of(col):                          # P(match | same actor)
    v = K[col]; both = hit = 0
    for i, j in same_actor_pairs:
        a, b = v.iloc[i], v.iloc[j]
        if pd.notna(a) and pd.notna(b):
            both += 1; hit += (a == b)
    return (hit / both if both else 0.0), both, hit

def u_of(col):                          # P(match | random pair) = sum (count/n)^2
    vc = K[col].dropna().value_counts(); n = int(vc.sum())
    return float(((vc / n) ** 2).sum())

print(f"labeled same-actor pairs (from disguise inboxes): {len(same_actor_pairs)}\n")
tab = []
for col in ["card", "device", "phone", "addr", "name", "ip_ua"]:
    m, both, hit = m_of(col); u = u_of(col)
    base = math.log2(m / u) if m > 0 else float("-inf")
    tab.append({"signal": col, "m=P(match|same)": round(m, 3),
                "u=P(match|rand)": f"{u:.2e}", "computed_BASE=log2(m/u)": round(base, 2)
                if base != float("-inf") else "-inf"})
comp = pd.DataFrame(tab).sort_values("computed_BASE=log2(m/u)", ascending=False,
                                     key=lambda s: s.map(lambda x: -1e9 if x == "-inf" else float(x)))
print(comp.to_string(index=False))
print("\nNOTE: name outranks card here (within-ring reuse), and ip_ua = -inf (never reused).")
print("We keep the MEANING-based BASE + per-value hub_penalty — see Appendix A above for why.")

labeled same-actor pairs (from disguise inboxes): 387

signal  m=P(match|same) u=P(match|rand) computed_BASE=log2(m/u)
  name            0.230        8.28e-05                   11.44
  card            0.167        1.04e-04                   10.65
 phone            0.044        8.53e-05                    9.02
device            0.058        1.71e-04                     8.4
  addr            0.018        6.74e-05                    8.07
 ip_ua            0.000        7.55e-05                    -inf

NOTE: name outranks card here (within-ring reuse), and ip_ua = -inf (never reused).
We keep the MEANING-based BASE + per-value hub_penalty — see Appendix A above for why.


## Appendix B — Full pipeline with COMPUTED base (Fellegi–Sunter + strong-gate)

Same graph pipeline, but BASE is now the data-derived `log₂(m/u)` from Appendix A instead of the meaning prior. One guard is mandatory: because computed name (11.4) outranks card and exceeds any threshold, pure computed weights would link strangers who share a rare name. So we add a **strong-signal gate** — an edge must include at least one *identity* field (email/card/device/phone); corroborate fields (addr/name/ip_ua) only add confidence. This is the disciplined way to honour "use computed base" without sacrificing precision. Below: run it end-to-end and compare to the deployed meaning-based model.

In [17]:
# --- computed BASE from the labeled m/u (Appendix A), reused live ---
BASE_C = {}
for s in ["email", "card", "phone", "device", "addr", "name", "ip_ua"]:
    m, _, _ = m_of(s); u = u_of(s)
    BASE_C[s] = math.log2(m / u) if m > 0 else 0.0     # ip_ua -> 0 (never reused = worthless)
BASE_C["email_alias"] = BASE_C["email"]                 # disguise = same inbox, same weight
STRONG_ID = {"email_alias", "email", "card", "device", "phone"}   # identity fields (gate)
TAU_C = 8.0                                             # a single strong field clears this
print("computed BASE:", {k: round(v, 2) for k, v in BASE_C.items()})

# --- rebuild edges with computed weights + strong-gate ---
pair_wc = defaultdict(float); pair_sc = defaultdict(list); has_strong = defaultdict(bool)

def add_c(sig, val, members):
    df = len(members)
    if df < 2 or (sig in HUB_CAP and df > HUB_CAP[sig]) or BASE_C[sig] <= 0:
        return
    w = BASE_C[sig] * hub_penalty(df, sig)
    for a, b in combinations(sorted(members), 2):
        pair_wc[(a, b)] += w; pair_sc[(a, b)].append((sig, round(w, 2)))
        if sig in STRONG_ID: has_strong[(a, b)] = True

for canon, members in email_members.items():           # email split (disguise vs plain)
    if len(members) < 2: continue
    sub = K.set_index("account_id").loc[members]
    sig = "email_alias" if (sub["email_aliasable"].all() and sub["email_raw"].nunique() > 1) else "email"
    add_c(sig, canon, members)
for sig, idx in index.items():                          # keyed signals (index built in Section 4)
    for val, members in idx.items():
        add_c(sig, val, members)

# gate: keep edge only if it clears TAU_C *and* carries a strong identity signal
ufc = UF(); kept_c = []
for (a, b), w in sorted(pair_wc.items()):
    if w >= TAU_C and has_strong[(a, b)]:
        ufc.union(a, b); kept_c.append((a, b, round(w, 2)))
comp_c = defaultdict(list)
for aid in K["account_id"]:
    if aid in ufc.p: comp_c[ufc.find(aid)].append(aid)
clusters_c = {r: sorted(m) for r, m in comp_c.items() if len(m) > 1}

linked_c = sum(len(m) for m in clusters_c.values())
print(f"\nCOMPUTED-base result : {len(clusters_c)} clusters, {linked_c} accounts linked")
print(f"MEANING-base result  : {len(clusters)} clusters, {sum(len(m) for m in clusters.values())} accounts linked")

# did the gate hold? any cluster with NO strong-identity signal would be a leak
leak = [r for r, m in clusters_c.items()
        if not any(has_strong[(a, b)] for a, b in combinations(m, 2) if (a, b) in has_strong)]
print(f"clusters lacking a strong-identity link (should be 0): {len(leak)}")

computed BASE: {'email': 13.8, 'card': 10.65, 'phone': 9.02, 'device': 8.4, 'addr': 8.07, 'name': 11.44, 'ip_ua': 0.0, 'email_alias': 13.8}



COMPUTED-base result : 692 clusters, 1521 accounts linked
MEANING-base result  : 692 clusters, 1519 accounts linked
clusters lacking a strong-identity link (should be 0): 0


In [18]:
# exactly which accounts differ between the two models, and why
meaning_linked = {a for m in clusters.values() for a in m}
computed_linked = {a for m in clusters_c.values() for a in m}
only_meaning = meaning_linked - computed_linked
only_computed = computed_linked - meaning_linked
print(f"linked by MEANING only : {len(only_meaning)}  -> {sorted(only_meaning)}")
print(f"linked by COMPUTED only: {len(only_computed)} -> {sorted(only_computed)}")

# for the meaning-only accounts, show they had NO strong-identity edge (why the gate drops them)
print("\nwhy the strong-gate drops them (their kept meaning-edges):")
for a, b, w, sigs in kept:
    if a in only_meaning or b in only_meaning:
        types = {s for s, _ in sigs}
        strong = types & {"email_alias", "email", "card", "device", "phone"}
        print(f"  {a}-{b}: signals={sorted(types)}  strong={sorted(strong) or 'NONE -> gated out'}")

linked by MEANING only : 0  -> []
linked by COMPUTED only: 2 -> ['A009781', 'A012677']

why the strong-gate drops them (their kept meaning-edges):


### Appendix B — ship the computed-base model as the primary deliverable

Computed base + strong-gate is the shipped model (Section 5's meaning-based run is kept as `*_meaning.*` for validation). Tier the computed clusters and write the primary `clusters.csv` / `edges.csv` / `findings.md`.

In [19]:
import csv
edge_sigs_c = defaultdict(set)
for a, b, w in kept_c:
    for s, _ in pair_sc[(a, b)]:
        edge_sigs_c[ufc.find(a)].add(s)

Kx = K.set_index("account_id")
def sval(members, col):
    vc = Kx.loc[members, col].dropna().value_counts()
    return vc.index[0] if len(vc) else None
colmap = {"email_alias":"email","email":"email","phone":"phone","card":"card",
          "device":"device","ip_ua":"ip_ua","addr":"addr","name":"name"}
STRONGSET = {"email","card","device","phone"}

rows_c = []
for r, members in clusters_c.items():
    sigs = edge_sigs_c[r]; n = len(members)
    has_alias = "email_alias" in sigs
    has_strong = bool(sigs & STRONGSET) or has_alias
    nt = len(sigs)
    tier = ("CERTAIN" if has_alias else "HIGH" if (has_strong and nt >= 2)
            else "MEDIUM" if has_strong else "LOW")
    if n > REVIEW_SIZE and not has_alias: tier = "REVIEW"
    max_w = max(w for a, b, w in kept_c if ufc.find(a) == r)
    dom = max(sigs, key=lambda s: BASE_C[s])
    conf = round(max_w * (1 + 0.4 * (nt - 1)), 2)
    links = "; ".join(f"{s}={sval(members, colmap[s])}" for s in sorted(sigs) if sval(members, colmap[s]))
    rows_c.append({"members": sorted(members), "size": n, "tier": tier, "confidence": conf,
                   "dominant": dom, "sigs": ",".join(sorted(sigs)), "links": links})
order = {"REVIEW":5,"CERTAIN":4,"HIGH":3,"MEDIUM":2,"LOW":1}
rows_c.sort(key=lambda c: (order[c["tier"]], c["confidence"], c["size"]), reverse=True)
for i, c in enumerate(rows_c, 1): c["cid"] = f"C{i:05d}"

OUT = Path("output"); OUT.mkdir(exist_ok=True)
with open(OUT / "clusters.csv", "w", newline="") as f:
    wr = csv.writer(f); wr.writerow(["account_id","cluster_id","cluster_size","confidence_tier","dominant_signal","linking_values"])
    for c in rows_c:
        for aid in c["members"]:
            wr.writerow([aid, c["cid"], c["size"], c["tier"], c["dominant"], c["links"]])
with open(OUT / "edges.csv", "w", newline="") as f:
    wr = csv.writer(f); wr.writerow(["account_a","account_b","combined_weight","signals"])
    for a, b, w in sorted(kept_c):
        wr.writerow([a, b, w, "|".join(f"{s}:{sw}" for s, sw in pair_sc[(a, b)])])
tc = Counter(c["tier"] for c in rows_c)
with open(OUT / "findings.md", "w") as f:
    f.write(f"# Ranked findings (PRIMARY: computed-base + strong-gate)\n\n"
            f"{N} accounts | {len(rows_c)} clusters | {sum(c['size'] for c in rows_c)} linked\n"
            f"Tiers: {dict(tc)}\n\n")
    for c in rows_c[:15]:
        f.write(f"### {c['cid']} — {c['size']} accts — {c['tier']} (conf {c['confidence']})\n"
                f"- Linked by: {c['links']}\n- Members: {', '.join(c['members'][:12])}"
                f"{' …' if c['size'] > 12 else ''}\n\n")
print("PRIMARY (computed-base) tiers:", dict(tc))
print("wrote clusters.csv, edges.csv, findings.md  (+ *_meaning.* validation copies)")


PRIMARY (computed-base) tiers: {'CERTAIN': 204, 'HIGH': 36, 'MEDIUM': 452}
wrote clusters.csv, edges.csv, findings.md  (+ *_meaning.* validation copies)
